# Stereo-Seq

## Import packages

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import harpy as hp
import spatialdata as sd
import pandas as pd
import numpy as np
import dask as da
import os
import gc
import geopandas as gpd
import matplotlib
from matplotlib import pyplot as plt
import seaborn as sns
import h5py
import json
import gc
import glob

from dask_image.imread import imread
import spatialdata_plot

from spatialdata.transformations import get_transformation, remove_transformation, set_transformation

/home/VIB.LOCAL/julien.mortier/.conda/envs/harpy/lib/python3.10/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [4]:
import sys
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(''))))
from utils.in_tissue import add_in_tissue, assign_region
from utils.plotting import best_datatree_scale, rel_fontsize
from utils.qc import qc_metrics
from _stereoseq import stereoseq

## Create sdata

In [5]:
dataset_id = "Demo_Mouse_Brain"
path_SAW = f"/data/groups/technologies/spatial.catalyst/Julien/SAW/demo_data/test_saw/{dataset_id}"
SN = "C04042E3"

regist_img_path = os.path.join(path_SAW, f"outs/image/{SN}_HE_regist.tif")
raw_h5ad_path = os.path.join(path_SAW, f"outs/feature_expression/{SN}.raw.h5ad")
tissue_cut_path = os.path.join(path_SAW, f"outs/image/{SN}_HE_tissue_cut.tif")
tissue_h5ad_path = [
    os.path.join(path_SAW, f"outs/feature_expression/{SN}.bin1.h5ad"),
    # os.path.join(path_SAW, f"outs/feature_expression/{SN}.bin5.h5ad"),
    # os.path.join(path_SAW, f"outs/feature_expression/{SN}.bin10.h5ad"),
    # os.path.join(path_SAW, f"outs/feature_expression/{SN}.bin20.h5ad"),
    os.path.join(path_SAW, f"outs/feature_expression/{SN}.bin50.h5ad"),
    # os.path.join(path_SAW, f"outs/feature_expression/{SN}.bin100.h5ad"),
    # os.path.join(path_SAW, f"outs/feature_expression/{SN}.bin150.h5ad"),
    # os.path.join(path_SAW, f"outs/feature_expression/{SN}.bin200.h5ad"),
]
output_path = f'/data/groups/technologies/spatial.catalyst/Julien/SAW/demo_data/test_saw/{dataset_id}/harpy/sdata.zarr'

barcodeToPos_path = f"/data/groups/technologies/spatial.catalyst/Julien/SAW/demo_data/input/mask/{SN}.barcodeToPos.h5"

pipeline_name = 'SAW'
species = 'mouse'
tissue_annotation_file = f'/data/groups/technologies/spatial.catalyst/Julien/SAW/demo_data/test_saw/Demo_Mouse_Brain/harpy/C04042E3_HE_regist.geojson'

In [ ]:
sdata = sd.read_zarr(output_path)
sdata

version mismatch: detected: RasterFormatV02, requested: FormatV04
/home/VIB.LOCAL/julien.mortier/.conda/envs/harpy/lib/python3.10/site-packages/zarr/creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)


SpatialData object, with associated Zarr store: /data/groups/technologies/spatial.catalyst/Julien/SAW/demo_data/test_saw/Demo_Mouse_Brain/harpy/sdata.zarr
├── Images
│     └── 'Demo_Mouse_Brain_image': DataTree[cyx] (3, 23520, 23520), (3, 11760, 11760), (3, 5880, 5880), (3, 2940, 2940), (3, 1470, 1470)
├── Labels
│     ├── 'Demo_Mouse_Brain_bin_1_labels': DataArray[yx] (18352, 17250)
│     ├── 'Demo_Mouse_Brain_bin_50_labels': DataArray[yx] (368, 346)
│     ├── 'Demo_Mouse_Brain_raw_labels': DataArray[yx] (23520, 23520)
│     ├── 'Demo_Mouse_Brain_tissue_cut': DataTree[yx] (23520, 23520), (11760, 11760), (5880, 5880), (2940, 2940), (1470, 1470)
│     └── 'Demo_Mouse_Brain_tissue_mask': DataArray[yx] (21494, 20466)
├── Shapes
│     ├── 'Demo_Mouse_Brain_bin_1_shapes': GeoDataFrame shape: (30565412, 1) (2D shapes)
│     ├── 'Demo_Mouse_Brain_bin_50_shapes': GeoDataFrame shape: (84163, 1) (2D shapes)
│     ├── 'Demo_Mouse_Brain_capture_area': GeoDataFrame shape: (1, 1) (2D shapes)
│     ├

: 

In [6]:
sdata = stereoseq(
    regist_img_path = regist_img_path,
    dataset_id = dataset_id,
    raw_h5ad_path = raw_h5ad_path,
    tissue_h5ad_path = tissue_h5ad_path,
    tissue_cut_path = tissue_cut_path,
    output = output_path,
)
sdata

2026-01-16 07:08:48.375 | WARNING  | harpy.io._spatial_data:_load_image_to_dask:231 - dims parameter is equal to ['c', 'y', 'x'], but will be ignored when reading in images from a file
/home/VIB.LOCAL/julien.mortier/.conda/envs/harpy/lib/python3.10/site-packages/zarr/creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
2026-01-16 07:09:06.957 | INFO     | harpy.image._manager:add_layer:47 - Writing results to layer 'Demo_Mouse_Brain_image'
2026-01-16 07:09:18.236 | INFO     | harpy.image._manager:add_layer:47 - Writing results to layer 'Demo_Mouse_Brain_tissue_cut'
/home/VIB.LOCAL/julien.mortier/.conda/envs/harpy/lib/python3.10/site-packages/spatialdata/models/models.py:1144: UserWarning: Converting `region_key: temporary_region_key` to categorical dtype.
  return convert_region_column_to_categorical(adata)


INFO     The instance_key column in 'table.obs' ('table.obs[temporary_instance_key]') will be relabeled to ensure a
         numeric data type, with a continuous range and without including the value 0 (which is reserved for the   
         background). The new labels will be stored in a new column named 'relabeled_temporary_instance_key'.      


/home/VIB.LOCAL/julien.mortier/.conda/envs/harpy/lib/python3.10/site-packages/spatialdata/_core/spatialdata.py:511: UserWarning: Converting `region_key: temporary_region_key` to categorical dtype.
  convert_region_column_to_categorical(table)
/home/VIB.LOCAL/julien.mortier/.conda/envs/harpy/lib/python3.10/site-packages/spatialdata/models/models.py:1144: UserWarning: Converting `region_key: temporary_region_key` to categorical dtype.
  return convert_region_column_to_categorical(adata)


INFO     The instance_key column in 'table.obs' ('table.obs[temporary_instance_key]') will be relabeled to ensure a
         numeric data type, with a continuous range and without including the value 0 (which is reserved for the   
         background). The new labels will be stored in a new column named 'relabeled_temporary_instance_key'.      


/home/VIB.LOCAL/julien.mortier/.conda/envs/harpy/lib/python3.10/site-packages/spatialdata/_core/spatialdata.py:511: UserWarning: Converting `region_key: temporary_region_key` to categorical dtype.
  convert_region_column_to_categorical(table)
/home/VIB.LOCAL/julien.mortier/.conda/envs/harpy/lib/python3.10/site-packages/spatialdata/models/models.py:1144: UserWarning: Converting `region_key: temporary_region_key` to categorical dtype.
  return convert_region_column_to_categorical(adata)


INFO     The instance_key column in 'table.obs' ('table.obs[temporary_instance_key]') will be relabeled to ensure a
         numeric data type, with a continuous range and without including the value 0 (which is reserved for the   
         background). The new labels will be stored in a new column named 'relabeled_temporary_instance_key'.      


/home/VIB.LOCAL/julien.mortier/.conda/envs/harpy/lib/python3.10/site-packages/spatialdata/_core/spatialdata.py:511: UserWarning: Converting `region_key: temporary_region_key` to categorical dtype.
  convert_region_column_to_categorical(table)


INFO     The Zarr backing store has been changed from None the new file path:                                      
         /data/groups/technologies/spatial.catalyst/Julien/SAW/demo_data/test_saw/Demo_Mouse_Brain/harpy/sdata.zarr


version mismatch: detected: RasterFormatV02, requested: FormatV04
/home/VIB.LOCAL/julien.mortier/.conda/envs/harpy/lib/python3.10/site-packages/zarr/creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)


SpatialData object, with associated Zarr store: /data/groups/technologies/spatial.catalyst/Julien/SAW/demo_data/test_saw/Demo_Mouse_Brain/harpy/sdata.zarr
├── Images
│     └── 'Demo_Mouse_Brain_image': DataTree[cyx] (3, 23520, 23520), (3, 11760, 11760), (3, 5880, 5880), (3, 2940, 2940), (3, 1470, 1470)
├── Labels
│     ├── 'Demo_Mouse_Brain_bin_1_labels': DataArray[yx] (18352, 17250)
│     ├── 'Demo_Mouse_Brain_bin_50_labels': DataArray[yx] (368, 346)
│     ├── 'Demo_Mouse_Brain_raw_labels': DataArray[yx] (23520, 23520)
│     └── 'Demo_Mouse_Brain_tissue_cut': DataTree[yx] (23520, 23520), (11760, 11760), (5880, 5880), (2940, 2940), (1470, 1470)
├── Shapes
│     ├── 'Demo_Mouse_Brain_bin_1_shapes': GeoDataFrame shape: (30565412, 1) (2D shapes)
│     ├── 'Demo_Mouse_Brain_bin_50_shapes': GeoDataFrame shape: (84163, 1) (2D shapes)
│     └── 'Demo_Mouse_Brain_raw_shapes': GeoDataFrame shape: (33419968, 1) (2D shapes)
└── Tables
      ├── 'Demo_Mouse_Brain_bin_1_table': AnnData (30565412, 2

In [ ]:
import dask 
with dask.config.set(scheduler="processes"):
    print("Starting rasterization...")
    sdata = hp.im.rasterize(
        sdata,
        shapes_layer=f"{dataset_id}_raw_shapes", 
        output_layer=f"{dataset_id}_raw_labels",
        chunks=512,
        overwrite=True,
    )
    sdata

Starting rasterization...


2026-01-17 13:58:51.359 | INFO     | harpy.image._manager:add_layer:47 - Writing results to layer 'Demo_Mouse_Brain_raw_labels'


In [7]:
levels = [level.split(f'{dataset_id}_')[1].split("_table")[0] for level in [*sdata.tables]]
print(f"{pipeline_name} levels: {levels}")

SAW levels: ['bin_1', 'bin_50', 'raw']


## Specify plotting parameters

In [8]:
DPI = 200
MAX_DIM_FOR_PLOTS = 10000

HE_image = f"{dataset_id}_image"

best_scale = best_datatree_scale(sdata[HE_image], MAX_DIM_FOR_PLOTS)
print(f'best scale: {best_scale}')

figsize_factor = DPI
figsize = (sdata.images[HE_image][best_scale]['image'].shape[2]/figsize_factor, sdata.images[HE_image][best_scale]['image'].shape[1]/figsize_factor)
print(f'figsize: {figsize}')

plots_path = os.path.join(os.path.dirname(output_path), 'plots')
os.makedirs(plots_path, exist_ok=True)
print(f'plots_path: {plots_path}')

best scale: scale2
figsize: (29.4, 29.4)
plots_path: /data/groups/technologies/spatial.catalyst/Julien/SAW/demo_data/test_saw/Demo_Mouse_Brain/harpy/plots


## Plot H&E

In [9]:
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = "H&E"

sdata.pl.render_images(
    element=HE_image, scale=best_scale
).pl.show(
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, 'HE.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

Plot saved at: /data/groups/technologies/spatial.catalyst/Julien/SAW/demo_data/test_saw/Demo_Mouse_Brain/harpy/plots/HE.png


## Plot SAW tissue segmentation mask

In [10]:
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = f"{pipeline_name} - Tissue segmentation mask"

sdata.pl.render_images(
    element=HE_image, scale=best_scale
).pl.render_labels(
    element=f"{dataset_id}_tissue_cut", fill_alpha=0.3, contour_px=rel_fontsize(fig, 0.003), outline_alpha=1, scale=best_scale
).pl.show(
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, f'{pipeline_name}_tissue_segmentation_mask.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

Plot saved at: /data/groups/technologies/spatial.catalyst/Julien/SAW/demo_data/test_saw/Demo_Mouse_Brain/harpy/plots/SAW_tissue_segmentation_mask.png


## Plot SAW levels

In [11]:
for level in levels:
    gc.collect()
    labels_layer = f"{dataset_id}_{level}_labels"

    fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

    title = f"{pipeline_name} - {level}"

    sdata.pl.render_images(
        element=HE_image, scale=best_scale
    ).pl.render_labels(
        element=labels_layer, fill_alpha=0.5, contour_px=5, outline_alpha=0, scale=best_scale
    ).pl.show(
        title=title,
        figsize=figsize,
        colorbar=False,
        dpi=DPI,
        ax=ax,
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    save_path = os.path.join(plots_path, f'{pipeline_name}_{level}.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path)

INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: /data/groups/technologies/spatial.catalyst/Julien/SAW/demo_data/test_saw/Demo_Mouse_Brain/harpy/plots/SAW_bin_1.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Julien/SAW/demo_data/test_saw/Demo_Mouse_Brain/harpy/plots/SAW_bin_50.png
INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: /data/groups/technologies/spatial.catalyst/Julien/SAW/demo_data/test_saw/Demo_Mouse_Brain/harpy/plots/SAW_raw.png


## Plot SAW QC metrics

In [12]:
# Plot QC metrics
for level in levels:
    gc.collect()
    labels_layer = f"{dataset_id}_{level}_labels"
    table_layer = f"{dataset_id}_{level}_table"

    for qc_metric in ['total_counts', 'n_genes_by_counts', 'pct_counts_mt']:
        values = sdata.tables[table_layer].obs[qc_metric].to_numpy()
        vmin, vmax = np.percentile(values, [1, 99])
        norm = matplotlib.colors.Normalize(vmin=vmin, vmax=vmax)
        
        fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

        title = f"{pipeline_name} - {qc_metric}"

        sdata.pl.render_images(
            element=HE_image, scale=best_scale
        ).pl.render_labels(
            element=labels_layer, fill_alpha=1, contour_px=5, outline_alpha=0, color=qc_metric, scale=best_scale, table_name=table_layer, cmap="viridis", norm=norm
        ).pl.show(
            title=title,
            figsize=figsize,
            colorbar=True,
            dpi=DPI,
            ax=ax,
        )

        ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
        ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
        ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
        ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

        cbar = fig.axes[-1]
        cbar.set_ylabel(qc_metric, rotation=270, labelpad=rel_fontsize(fig, 0.025), fontsize = rel_fontsize(fig, 0.02))
        cbar.tick_params(axis="y", which="both", labelsize=rel_fontsize(fig, 0.015))

        save_path = os.path.join(plots_path, f'{pipeline_name}_{qc_metric}_{level}.png')
        fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

        plt.close(fig)

        print("Plot saved at:", save_path)

INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: /data/groups/technologies/spatial.catalyst/Julien/SAW/demo_data/test_saw/Demo_Mouse_Brain/harpy/plots/SAW_total_counts_bin_1.png
INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: /data/groups/technologies/spatial.catalyst/Julien/SAW/demo_data/test_saw/Demo_Mouse_Brain/harpy/plots/SAW_n_genes_by_counts_bin_1.png
INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: /data/groups/technologies/spatial.catalyst/Julien/SAW/demo_data/test_saw/Demo_Mouse_Brain/harpy/plots/SAW_pct_counts_mt_bin_1.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Julien/SAW/demo_data/test_saw/Demo_Mouse_Brain/harpy/plots/SAW_total_counts_bin_50.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Julien/SAW/

## Calculate pixel size

In [13]:
microns_per_pixel = 0.500
print(f"Pixel size: {microns_per_pixel:.3f} µm")

Pixel size: 0.500 µm


## Create bounding box for total capture area

In [14]:
# Add bounding box for total capture area
import shapely

geoms = sdata.shapes[f"{dataset_id}_raw_shapes"].geometry
bbox = shapely.MultiPolygon(list(geoms)).minimum_rotated_rectangle

sdata = hp.sh.add_shapes_layer(
    sdata,
    input = gpd.GeoDataFrame(geometry=[bbox]),
    output_layer = f"{dataset_id}_capture_area",
    transformations = get_transformation(sdata.shapes[f"{dataset_id}_raw_shapes"], get_all=True),
    overwrite = True,
)

In [15]:
# Export bbox GeoJSON
sdata.shapes[f'{dataset_id}_capture_area'].to_file(os.path.join(os.path.dirname(output_path), f'{dataset_id}_capture_area.geojson'), driver="GeoJSON")

/home/VIB.LOCAL/julien.mortier/.conda/envs/harpy/lib/python3.10/site-packages/pyogrio/geopandas.py:917: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(


In [ ]:
# Check capture area bbox
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = "Capture area bounding box"

sdata.pl.render_images(
    element=HE_image, scale=best_scale
).pl.render_shapes(
    element=f"{dataset_id}_bin_10_shapes", color='black', fill_alpha=0.5, outline_width=rel_fontsize(fig, 0.0012), outline_alpha=1, outline_color='black'
).pl.render_shapes(
    element=f'{dataset_id}_capture_area', color='#1f77b4', fill_alpha=0.1, outline_width=rel_fontsize(fig, 0.002), outline_alpha=1, outline_color='red'
).pl.show(
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
    pad_extent=rel_fontsize(fig, 0.03)
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, 'capture_area_bbox.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

INFO     Value for parameter 'color' appears to be a color, using it as such.                                      
INFO     Value for parameter 'color' appears to be a color, using it as such.                                      
INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                
Plot saved at: /data/groups/technologies/spatial.catalyst/Julien/SAW/demo_data/test_saw/Demo_Mouse_Brain/harpy/plots/capture_area_bbox.png


In [19]:
# Area of capture area in mm²
gdf = sdata.shapes[f'{dataset_id}_capture_area']
capture_area = (gdf.geometry.area.sum() * microns_per_pixel**2) / 1000000
print(f"Area of capture area: {capture_area:.2f} mm²")

Area of capture area: 138.29 mm²


In [20]:
from math import dist
coords = list(bbox.exterior.coords)

# Compute lengths of bbox
side1 = dist(coords[0], coords[1]) * microns_per_pixel / 10000
side2 = dist(coords[1], coords[2]) * microns_per_pixel / 10000

print(f"Dimensions of capture area bbox: {side1:.2f} cm x {side2:.2f} cm")

Dimensions of capture area bbox: 1.18 cm x 1.18 cm


## Add tissue annotation

In [21]:
# Add tissue annotation as shapes layer and as labels layer
gdf = gpd.read_file(tissue_annotation_file)
gdf = gdf.explode(index_parts=False) # This avoid problems with MultiPolygon geometries
gdf.index = gdf.index + 1 # Index 0 is not allowed

sdata = hp.sh.add_shapes_layer(
    sdata,
    input = gdf,
    output_layer = f'{dataset_id}_tissue',
    transformations = get_transformation(sdata.images[f"{dataset_id}_image"], get_all=True),
    overwrite = True,
)

sdata = hp.im.rasterize(
    sdata,
    shapes_layer = f'{dataset_id}_tissue',
    output_layer = f'{dataset_id}_tissue_mask',
    chunks = 1024,
    overwrite = True,
)

2026-01-16 08:37:55.308 | INFO     | harpy.image._manager:add_layer:47 - Writing results to layer 'Demo_Mouse_Brain_tissue_mask'


In [22]:
# Check tissue annotation
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = "Tissue annotation"

sdata.pl.render_images(
    element=HE_image, scale=best_scale
).pl.render_shapes(
    element=f'{dataset_id}_tissue', color='#1f77b4', fill_alpha=0.3, outline_width=rel_fontsize(fig, 0.002), outline_alpha=1, outline_color='red'
).pl.show(
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
    pad_extent=rel_fontsize(fig, 0.02)
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, 'qupath_tissue_annotation.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

INFO     Value for parameter 'color' appears to be a color, using it as such.                                      
Plot saved at: /data/groups/technologies/spatial.catalyst/Julien/SAW/demo_data/test_saw/Demo_Mouse_Brain/harpy/plots/qupath_tissue_annotation.png


In [23]:
# Determine which spots are in tissue annotation
for level in levels:
    gc.collect()
    sdata = add_in_tissue(
        sdata, 
        table_layer = f"{dataset_id}_{level}_table", 
        tissue_mask = sdata.labels[f'{dataset_id}_tissue_mask'].compute().data
    )

2026-01-16 08:46:19.043 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'Demo_Mouse_Brain_bin_1_table' already exists. Overwriting...
2026-01-16 08:52:26.571 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'Demo_Mouse_Brain_bin_50_table' already exists. Overwriting...
2026-01-16 09:03:03.299 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'Demo_Mouse_Brain_raw_table' already exists. Overwriting...


In [26]:
# Check spots in tissue
for level in levels:
    gc.collect()
    labels_layer = f"{dataset_id}_{level}_labels"
    table_layer = f"{dataset_id}_{level}_table"

    fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

    title = "Tissue annotation spots"

    sdata.pl.render_images(
        element=HE_image, scale=best_scale
    ).pl.render_labels(
        element=labels_layer, fill_alpha=0.5, contour_px=5, outline_alpha=0, color='in_tissue_annotation', table_name=table_layer, scale=best_scale
    ).pl.show(
        title=title,
        figsize=figsize,
        colorbar=False,
        dpi=DPI,
        ax=ax,
        pad_extent=rel_fontsize(fig, 0.02)
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    save_path = os.path.join(plots_path, f'qupath_annotation_spots_{level}.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path)

INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: /data/groups/technologies/spatial.catalyst/Julien/SAW/demo_data/test_saw/Demo_Mouse_Brain/harpy/plots/qupath_annotation_spots_bin_1.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Julien/SAW/demo_data/test_saw/Demo_Mouse_Brain/harpy/plots/qupath_annotation_spots_bin_50.png
INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: /data/groups/technologies/spatial.catalyst/Julien/SAW/demo_data/test_saw/Demo_Mouse_Brain/harpy/plots/qupath_annotation_spots_raw.png


In [ ]:
tissue_mask = sdata.labels[f'{dataset_id}_tissue_mask'].compute().data
mask_shape = tissue_mask.shape

barcode_rows = []
chunk_size = 2048

with h5py.File(barcodeToPos_path, "r") as f:
    dset = f["bpMatrix_1"]
    n_y, n_x, _ = dset.shape

    for y_start in range(0, n_y, chunk_size):
        y_end = min(y_start + chunk_size, n_y)
        slab = dset[y_start:y_end, :, 0] 
        ys, xs = np.nonzero(slab)
        ys = ys + y_start

        df_chunk = pd.DataFrame({
            "x": xs,
            "y": ys,
        })

        df_chunk["barcode"] = df_chunk["x"].astype(str) + "_" + df_chunk["y"].astype(str)

        df_chunk["in_tissue_annotation"] = df_chunk.apply(
            lambda row: assign_region((row["x"], row["y"]), tissue_mask, tissue_mask.shape), axis=1
        )

        barcode_rows.append(df_chunk)

barcode_df = pd.concat(barcode_rows, ignore_index=True)

# Total number of spots in capture area
print(f"Shape of barcode whitelist: ({n_y}, {n_x})")
print(f"Total number of possible coordinates: {n_y * n_x:,}") # NOTE: This includes track lines
print(f"Total number of spots in capture area: {len(barcode_df):,}")

# Total number of spots in tissue
total_n_spots_in_tissue = len(barcode_df[barcode_df['in_tissue_annotation'] == 1])
print(f"Total number of spots in tissue: {total_n_spots_in_tissue:,}")

Shape of barcode whitelist: (23522, 23522)
Total number of possible coordinates: 553,284,484
Total number of spots in capture area: 441,033,659
Total number of spots in tissue: 150035900


In [ ]:
# Total number of spots in table
n_spots = len(sdata.tables[f"{dataset_id}_raw_table"].obs)
print(f"Number of spots in table: {n_spots:,}")

Number of spots in table: 33419968


In [ ]:
# Total number of spots in table with at least one UMI
n_spots_not_zero = len(sdata.tables[f"{dataset_id}_raw_table"].obs[sdata.tables[f"{dataset_id}_raw_table"].X.sum(axis=1) > 0])
print(f"Number of spots in table with at least one UMI: {n_spots_not_zero:,}")

Number of spots in table with at least one UMI: 33419968


In [ ]:
# Number of spots in tissue
n_spots_in_tissue = len(sdata.tables[f"{dataset_id}_raw_table"][sdata.tables[f"{dataset_id}_raw_table"].obs['in_tissue_annotation'] == 1].obs)
print(f"Number of spots in tissue: {n_spots_in_tissue:,}")

Number of spots in tissue: 28725138


In [ ]:
# Number of spots in tissue with at least one UMI
in_tissue_mask = sdata.tables[f"{dataset_id}_raw_table"].obs['in_tissue_annotation'] == 1
umis_mask = np.array(sdata.tables[f"{dataset_id}_raw_table"].X.sum(axis=1)).flatten() > 0
spots_mask = in_tissue_mask & umis_mask
n_spots_in_tissue_not_zero = spots_mask.sum()
print(f"Number of spots in tissue with at least one UMI: {n_spots_in_tissue_not_zero:,}")

Number of spots in tissue with at least one UMI: 28725138


In [33]:
# Save barcodes
with open(os.path.join(os.path.dirname(output_path), "all_barcodes.txt"), "w") as f:
    for item in barcode_df['barcode']:
        f.write(f"{item}\n")
        
with open(os.path.join(os.path.dirname(output_path), "barcodes_in_tissue.txt"), "w") as f:
    for item in barcode_df[barcode_df['in_tissue_annotation'] == 1]['barcode']:
        f.write(f"{item}\n")      

In [34]:
# Area of tissue annotation that overlaps with capture area in mm²
tissue_gdf = sdata.shapes[f"{dataset_id}_tissue"].unary_union
capture_gdf = sdata.shapes[f"{dataset_id}_capture_area"].unary_union

overlap_geom = tissue_gdf.intersection(capture_gdf)

overlap_area_mm2 = (overlap_geom.area * microns_per_pixel**2) / 1000000

print(f"Capture area covered by tissue: {overlap_area_mm2:.2f} mm²")

Capture area covered by tissue: 46.95 mm²


## Capture effeciency metrics

In [42]:
# Total number of genes
print(f"Total number of genes: {(sdata.tables[f'{dataset_id}_raw_table'].X.sum(axis=0) > 0).sum():,}")

Total number of genes: 25,110


In [41]:
# Number of genes in tissue
print(f"Number of genes in tissue: {(sdata.tables[f'{dataset_id}_raw_table'][sdata.tables[f'{dataset_id}_raw_table'].obs['in_tissue_annotation'] == 1].X.sum(axis=0) > 0).sum():,}")

Number of genes in tissue: 24,743


In [40]:
# Total number of unique UMIs
print(f"Total number of unique UMIs: {sdata.tables[f'{dataset_id}_raw_table'].X.sum():,}")

Total number of unique UMIs: 42,497,896


In [43]:
# Number of unique UMIs in tissue
print(f"Number of unique UMIs in tissue: {sdata.tables[f'{dataset_id}_raw_table'][sdata.tables[f'{dataset_id}_raw_table'].obs['in_tissue_annotation'] == 1].X.sum():,}")

Number of unique UMIs in tissue: 37,145,992


In [44]:
raw_barcode_gene_exp = pd.read_csv(os.path.join(path_SAW, f'outs/feature_expression/{SN}_raw_barcode_gene_exp.txt'), delimiter='\t')
# NOTE: An annotated list file with the information of coordinate, gene, MID, read counts, which is prepared to be a sampling file that performs sequence saturation.

print(f"Total number of UMIs: {raw_barcode_gene_exp['readCount'].sum():,}")
unique_UMIs = raw_barcode_gene_exp[['x', 'y', 'geneID', 'MIDIndex']].drop_duplicates().shape[0]
print(f"Total number of unique UMIs: {unique_UMIs:,}")

Total number of UMIs: 51,167,257
Total number of unique UMIs: 42,497,896
